In [2]:
# 1. Imports + load data
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error

train_path = r"C:\Users\sswap\OneDrive\Desktop\Swapnil\University\UNSW Masters\ZZSC9020 - Data Science Project\Data\nsw_train.csv"
val_path   = r"C:\Users\sswap\OneDrive\Desktop\Swapnil\University\UNSW Masters\ZZSC9020 - Data Science Project\Data\nsw_validation.csv"
test_path  = r"C:\Users\sswap\OneDrive\Desktop\Swapnil\University\UNSW Masters\ZZSC9020 - Data Science Project\Data\nsw_test.csv"

train_df = pd.read_csv(train_path, parse_dates=["DATETIME"])
val_df   = pd.read_csv(val_path, parse_dates=["DATETIME"])
test_df  = pd.read_csv(test_path, parse_dates=["DATETIME"])

df = pd.concat([train_df, val_df]).sort_values("DATETIME").reset_index(drop=True)

In [4]:
# 2. Helper function to build Prophet with regressors
def build_prophet_model(feature_cols):
    m = Prophet(
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality=True
    )
    for col in feature_cols:
        m.add_regressor(col)
    return m

In [6]:
# 3. Helper function to train Prophet on 2010–2018 and predict 2019
def run_prophet(df, test_df, feature_cols, model_name, output_path):
    df_prophet = df.rename(columns={"DATETIME": "ds", "TOTALDEMAND": "y"})
    test_prophet = test_df.rename(columns={"DATETIME": "ds"})

    # Train on all data up to end of 2018
    train_full = df_prophet[df_prophet["ds"].dt.year <= 2018]

    m = build_prophet_model(feature_cols)
    m.fit(train_full)

    # Prepare future dataframe
    future = test_prophet[["ds"] + feature_cols]

    # Forecast
    forecast = m.predict(future)

    # Build output CSV
    output = pd.DataFrame({
        "DATETIME": test_df["DATETIME"],
        model_name: forecast["yhat"]
    })

    output.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

In [8]:
# 4. Define feature set (include radiation_day_before)
TARGET = "TOTALDEMAND"

DROP_COLS_WITH = [
    "DATETIME",
    TARGET,
    "TEMPERATURE",
    "radiation",
    "forecast_closest",
    "forecast_12hr_prior",
    "forecast_dayprior"
]

FEATURES_WITH = [c for c in df.columns if c not in DROP_COLS_WITH]

In [11]:
# 5. Run Prophet (with radiation_day_before)
run_prophet(
    df=df,
    test_df=test_df,
    feature_cols=FEATURES_WITH,
    model_name="prophet",
    output_path=r"C:\Users\sswap\OneDrive\Desktop\Swapnil\University\UNSW Masters\ZZSC9020 - Data Science Project\Data\prophet_predictions.csv"
)

17:28:00 - cmdstanpy - INFO - Chain [1] start processing
17:29:58 - cmdstanpy - INFO - Chain [1] done processing


Saved: C:\Users\sswap\OneDrive\Desktop\Swapnil\University\UNSW Masters\ZZSC9020 - Data Science Project\Data\prophet_predictions.csv


In [12]:
# 6. Define feature set (exclude radiation_day_before)
DROP_COLS_WITHOUT = [
    "radiation_day_before",
    "DATETIME",
    TARGET,
    "TEMPERATURE",
    "radiation",
    "forecast_closest",
    "forecast_12hr_prior",
    "forecast_dayprior"
]

FEATURES_WITHOUT = [c for c in df.columns if c not in DROP_COLS_WITHOUT]

In [14]:
# 7. Run Prophet (without radiation_day_before)
run_prophet(
    df=df,
    test_df=test_df,
    feature_cols=FEATURES_WITHOUT,
    model_name="prophet",
    output_path=r"C:\Users\sswap\OneDrive\Desktop\Swapnil\University\UNSW Masters\ZZSC9020 - Data Science Project\Data\prophet_predictions_without_radiation.csv"
)

17:32:43 - cmdstanpy - INFO - Chain [1] start processing
17:35:10 - cmdstanpy - INFO - Chain [1] done processing


Saved: C:\Users\sswap\OneDrive\Desktop\Swapnil\University\UNSW Masters\ZZSC9020 - Data Science Project\Data\prophet_predictions_without_radiation.csv


In [17]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

pred_with = pd.read_csv(
    r"C:\Users\sswap\OneDrive\Desktop\Swapnil\University\UNSW Masters\ZZSC9020 - Data Science Project\Data\prophet_predictions.csv"
)

# FIX: European date format → dayfirst=True
pred_with["DATETIME"] = pd.to_datetime(pred_with["DATETIME"], dayfirst=True)

test_df = pd.read_csv(
    r"C:\Users\sswap\OneDrive\Desktop\Swapnil\University\UNSW Masters\ZZSC9020 - Data Science Project\Data\nsw_test.csv",
    parse_dates=["DATETIME"]
)

df_with = test_df.merge(pred_with, on="DATETIME")

y_true = df_with["TOTALDEMAND"]
y_pred = df_with["prophet"]

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

ss_res = np.sum((y_true - y_pred)**2)
ss_tot = np.sum((y_true - np.mean(y_true))**2)
r2 = 1 - ss_res/ss_tot

print("Metrics for Prophet WITH radiation_day_before:")
print("RMSE:", rmse)
print("MAE:", mae)
print("MAPE (%):", mape)
print("R²:", r2)

Metrics for Prophet WITH radiation_day_before:
RMSE: 520.9564430043939
MAE: 367.33513132996575
MAPE (%): 4.539006893422947
R²: 0.8262626415899659


In [19]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Load predictions WITHOUT radiation_day_before
pred_without = pd.read_csv(
    r"C:\Users\sswap\OneDrive\Desktop\Swapnil\University\UNSW Masters\ZZSC9020 - Data Science Project\Data\prophet_predictions_without_radiation.csv"
)

# Fix date format
pred_without["DATETIME"] = pd.to_datetime(pred_without["DATETIME"], dayfirst=True)

# Load actual demand
test_df = pd.read_csv(
    r"C:\Users\sswap\OneDrive\Desktop\Swapnil\University\UNSW Masters\ZZSC9020 - Data Science Project\Data\nsw_test.csv",
    parse_dates=["DATETIME"]
)

# Merge
df_without = test_df.merge(pred_without, on="DATETIME")

# Use the correct column name
y_true = df_without["TOTALDEMAND"]
y_pred = df_without["prophet_no_radiation_day_before"]

# Metrics
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

ss_res = np.sum((y_true - y_pred)**2)
ss_tot = np.sum((y_true - np.mean(y_true))**2)
r2 = 1 - ss_res/ss_tot

print("Metrics for Prophet WITHOUT radiation_day_before:")
print("RMSE:", rmse)
print("MAE:", mae)
print("MAPE (%):", mape)
print("R²:", r2)

Metrics for Prophet WITHOUT radiation_day_before:
RMSE: 521.2071156235596
MAE: 367.7735066410959
MAPE (%): 4.544028586398518
R²: 0.8260954042817525
